# Baseline: ChronoFormer (mortality risk)

Sequence of clinical **condition codes** + **inter-event time gaps** → binary label (death). Data: Hugging Face `synthea-575k-patients` (parquet). Next project steps: attention rollout / saliency (see proposal).


### Environment
Which Python binary runs this notebook (useful for matching `pip install`).


In [1]:
import sys
print(sys.executable)


c:\Users\belya\AppData\Local\Programs\Python\Python310\python.exe


### Imports, hyperparameters, device
Libraries, `CONFIG` (seq length, batch, model width, epochs, LR). **CUDA**: `cudnn.benchmark`, `pin_memory`, `DataLoader(num_workers=0)` for reliable behavior on Windows/Jupyter with large in-memory datasets.


In [2]:
import polars as pl
import torch
import torch.nn as nn
import math
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from tqdm.auto import tqdm

CONFIG = {
    "max_len": 256,
    "batch_size": 128,
    "d_model": 128,
    "n_heads": 4,
    "epochs": 50,
    "lr": 5e-5,
    "patience": 5,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

print(f"Using device: {CONFIG['device']}")
if CONFIG["device"].type == "cuda":
    torch.backends.cudnn.benchmark = True

_pin = CONFIG["device"].type == "cuda"
_loader_kw = dict(num_workers=0, pin_memory=_pin)


Using device: cuda


### Raw data loading
**Patients**: death date → label. **Conditions**: sorted events per patient, day deltas between consecutive visits. Train/val/test split 80/10/10.


In [3]:
def get_full_meds_data():
    base_url = "hf://datasets/richardyoung/synthea-575k-patients/data"
    print("Loading patients...")
    patients_lf = pl.scan_parquet(f"{base_url}/patients.parquet").select([
        pl.col("Id").alias("patient_id"),
        pl.col("DEATHDATE")
    ])
    labels_df = patients_lf.with_columns(
        pl.when(pl.col("DEATHDATE").is_not_null()).then(1).otherwise(0).alias("label")
    ).collect()
    all_ids = labels_df["patient_id"].to_list()
    train_ids, temp_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
    val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)
    print(f"Split: train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}")
    print("Loading events (slow)...")
    events_lf = pl.scan_parquet(f"{base_url}/conditions.parquet").select([
        pl.col("PATIENT").alias("patient_id"),
        pl.col("START").str.to_datetime().alias("timestamp"),
        pl.col("CODE").alias("event_code")
    ])
    events_df = events_lf.sort(["patient_id", "timestamp"]).with_columns(
        pl.col("timestamp").diff().dt.total_days().over("patient_id").fill_null(0).alias("time_delta")
    ).collect()
    return labels_df, events_df, (train_ids, val_ids, test_ids)


def create_vocab(events_df):
    unique_codes = events_df["event_code"].unique().to_list()
    vocab = {code: i + 2 for i, code in enumerate(unique_codes)}
    vocab[''] = 0
    vocab["[UNK]"] = 1
    return vocab


### PyTorch `Dataset`
Group events by patient, **tokenize** codes via vocab (once per split), **pad or truncate** to `max_len`, boolean mask for valid positions.


In [4]:
class MEDSDataset(Dataset):
    def __init__(self, events_df, labels_df, vocab, target_ids, max_len=256, desc="Dataset"):
        self.max_len = max_len
        subset_labels = labels_df.filter(pl.col("patient_id").is_in(target_ids))
        self.label_map = dict(zip(subset_labels["patient_id"], subset_labels["label"]))
        print(f"Aggregate {desc}...")
        subset_events = events_df.filter(pl.col("patient_id").is_in(target_ids))
        grouped = (subset_events.group_by("patient_id", maintain_order=True)
                   .agg([pl.col("event_code"), pl.col("time_delta")]))
        self.patient_ids = grouped["patient_id"].to_list()
        raw_codes = grouped["event_code"].to_list()
        self.times = grouped["time_delta"].to_list()
        print(f"Tokenize {desc}...")
        self.codes = [[vocab.get(c, 1) for c in row] for row in tqdm(raw_codes, desc=desc)]

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        p_id = self.patient_ids[idx]
        codes = list(self.codes[idx])
        times = list(self.times[idx])
        if len(codes) > self.max_len:
            codes, times = codes[-self.max_len:], times[-self.max_len:]
            mask = [1] * self.max_len
        else:
            pad_len = self.max_len - len(codes)
            mask = [1] * len(codes) + [0] * pad_len
            codes += [0] * pad_len
            times += [0.0] * pad_len
        return {
            "x": torch.tensor(codes, dtype=torch.long),
            "times": torch.tensor(times, dtype=torch.float),
            "mask": torch.tensor(mask, dtype=torch.bool),
            "label": torch.tensor(self.label_map[p_id], dtype=torch.float)
        }


### ChronoFormer (proposal core)
**TimeAwareAttention**: add learnable time bias to attention logits. **ChronoFormer**: event + time embeddings → attention → layer norm → **last token** → MLP + sigmoid.


In [5]:
class TimeAwareAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_time_bins=1000):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.time_proj = nn.Embedding(max_time_bins, n_heads)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, time_deltas, mask=None):
        b, s, d = x.size()
        q = self.q(x).view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k(x).view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v(x).view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        t_bins = torch.clamp(time_deltas.long(), 0, 999)
        time_bias = self.time_proj(t_bins).transpose(1, 2).unsqueeze(2)
        assert scores.shape == (b, self.n_heads, s, s), f"Bad scores shape: {scores.shape}"
        scores = scores + time_bias
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        return self.out(context)


class ChronoFormer(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4):
        super().__init__()
        self.event_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.time_embedding = nn.Embedding(1000, d_model)
        self.attention = TimeAwareAttention(d_model, n_heads)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x, time_deltas, mask=None):
        t_bins = torch.clamp(time_deltas.long(), 0, 999)
        x = self.event_embedding(x) + self.time_embedding(t_bins)
        x = self.attention(x, time_deltas, mask)
        x = self.norm(x)
        return self.classifier(x[:, -1, :])


### Training / evaluation
Build loaders, **AdamW + BCE**, track val **AUROC** (save best weights), early stopping, then **test** AUROC/F1. Writes `chronoformer_best.pt` in the working directory.


In [6]:
def run_experiment():
    labels_df, events_df, splits = get_full_meds_data()
    vocab = create_vocab(events_df)
    train_ids, val_ids, test_ids = splits
    ml = CONFIG["max_len"]
    train_loader = DataLoader(
        MEDSDataset(events_df, labels_df, vocab, train_ids, max_len=ml, desc="train"),
        batch_size=CONFIG["batch_size"], shuffle=True, **_loader_kw)
    val_loader = DataLoader(
        MEDSDataset(events_df, labels_df, vocab, val_ids, max_len=ml, desc="val"),
        batch_size=CONFIG["batch_size"], **_loader_kw)
    test_loader = DataLoader(
        MEDSDataset(events_df, labels_df, vocab, test_ids, max_len=ml, desc="test"),
        batch_size=CONFIG["batch_size"], **_loader_kw)
    model = ChronoFormer(len(vocab), CONFIG["d_model"], CONFIG["n_heads"]).to(CONFIG["device"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
    criterion = nn.BCELoss()
    best_auroc = 0.0
    patience_counter = 0
    dev = CONFIG["device"]
    print("Training...")
    saw_first_batch = False
    for epoch in range(CONFIG["epochs"]):
        model.train()
        train_pbar = tqdm(train_loader, desc=f"ep{epoch} train", leave=False)
        total_loss = 0.0
        for batch in train_pbar:
            if not saw_first_batch:
                print("First batch.")
                saw_first_batch = True
            x, t, m, y = [batch[k].to(dev, non_blocking=_pin) for k in ["x", "times", "mask", "label"]]
            optimizer.zero_grad()
            out = model(x, t, m).squeeze()
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            train_pbar.set_postfix(loss=loss.item())
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"ep{epoch} val", leave=False):
                x, t, m, y = [batch[k].to(dev, non_blocking=_pin) for k in ["x", "times", "mask", "label"]]
                out = model(x, t, m).squeeze()
                preds.extend(out.cpu().numpy())
                targets.extend(y.cpu().numpy())
        if len(set(targets)) < 2:
            auroc = float("nan")
        else:
            auroc = roc_auc_score(targets, preds)
        f1 = f1_score(targets, [1 if p > 0.5 else 0 for p in preds])
        print(f"ep{epoch} loss={total_loss/len(train_loader):.4f} auroc={auroc:.4f} f1={f1:.4f}")
        if auroc != auroc:
            patience_counter += 1
        elif auroc > best_auroc:
            best_auroc = auroc
            torch.save(model.state_dict(), "chronoformer_best.pt")
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print("Early stop.")
            break
    print("Test...")
    model.load_state_dict(torch.load("chronoformer_best.pt"))
    model.eval()
    test_preds, test_targets = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="test"):
            x, t, m, y = [batch[k].to(dev, non_blocking=_pin) for k in ["x", "times", "mask", "label"]]
            out = model(x, t, m).squeeze()
            test_preds.extend(out.cpu().numpy())
            test_targets.extend(y.cpu().numpy())
    if len(set(test_targets)) < 2:
        final_auroc = float("nan")
    else:
        final_auroc = roc_auc_score(test_targets, test_preds)
    final_f1 = f1_score(test_targets, [1 if p > 0.5 else 0 for p in test_preds])
    print(f"test auroc={final_auroc:.4f} f1={final_f1:.4f}")


def dry_run_test():
    print("Dry run...")
    vocab_size = 100
    model = ChronoFormer(vocab_size, d_model=128, n_heads=4).to(CONFIG["device"])
    dummy_x = torch.randint(0, vocab_size, (CONFIG["batch_size"], CONFIG["max_len"])).to(CONFIG["device"])
    dummy_times = torch.randn(CONFIG["batch_size"], CONFIG["max_len"]).to(CONFIG["device"])
    dummy_mask = torch.ones(CONFIG["batch_size"], CONFIG["max_len"]).to(CONFIG["device"])
    out = model(dummy_x, dummy_times, dummy_mask)
    print(f"ok {out.shape}")


### Sanity check
One forward pass with random integers (shape `[batch, max_len]`). Run before full training.


In [7]:
dry_run_test()


Dry run...
ok torch.Size([128, 1])


### Full experiment
Loads all data, tokenizes three splits, trains until early stop, evaluates on test. **Long runtime** and high RAM. Run when the cells above have executed without error.


In [8]:
run_experiment()


Loading patients...
Split: train=460332 val=57541 test=57542
Loading events (slow)...
Aggregate train...
Tokenize train...


train:   0%|          | 0/460332 [00:00<?, ?it/s]

Aggregate val...
Tokenize val...


val:   0%|          | 0/57541 [00:00<?, ?it/s]

Aggregate test...
Tokenize test...


...

ep21 val:   0%|          | 0/450 [00:00<?, ?it/s]

ep21 loss=0.1525 auroc=0.9555 f1=0.7334
Early stop.
Test...


C:\Users\belya\AppData\Local\Temp\ipykernel_31864\294745183.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("chronoformer_best.pt"))


test:   0%|          | 0/450 [00:00<?, ?it/s]

test auroc=0.9553 f1=0.7353
